# 基于MindSpore的DistilGPT-2语言模型微调与文本生成任务

## 案例介绍

本 Notebook 演示如何在 **MindSpore + MindSpore NLP** 生态中，对 **Causal LM（自回归语言模型）** 进行微调，并在训练完成后进行文本生成（续写）推理。

示例流程包含：

- 环境与依赖准备（版本检查、可选安装）
- 数据集加载：Wikitext-2-raw-v1
- 预训练模型加载：DistilGPT-2
- 文本预处理与语言建模样本构造（shift labels、padding、batch）
- 训练与验证（loss 监控）
- 推理生成与模型保存/加载（可选）


## 模型简介

我们选用 HuggingFace 社区中体量较小、易于在单机设备上快速实验的 **DistilGPT-2** 模型：

- 模型 ID：`distilgpt2`
- 结构：GPT-2 的轻量版，自回归语言模型（Causal LM）
- 任务：给定前文，预测下一个 token（下一词/子词）

MindSpore NLP 的 `AutoTokenizer` 和 `AutoModelForCausalLM` 提供与 HuggingFace Transformers 类似的使用方式。


## 环境准备

本案例推荐运行环境（示例）：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :---------------- |
| 3.10   | 2.7.0     | 0.5.1             |

运行设备建议：Ascend（如 Atlas 系列）。如你使用在线算力平台/预置镜像环境，通常无需重新安装 MindSpore，仅需按需补齐 MindSpore NLP 及辅助依赖。


In [ ]:
# 检查 mindspore / MindSpore NLP 版本（若未安装，可先跳过执行）
!pip show mindspore
!pip show mindnlp


如果当前环境缺少依赖或版本不匹配，可参考下方（已注释）安装命令按需安装：

In [ ]:
# 如果环境已经安装了指定版本的 mindspore 和 MindSpore NLP，可以跳过本单元格
# 在昇思 AI 实验室 / Atlas 服务器上建议使用镜像自带的 MindSpore，再手动安装 MindSpore NLP。

# 安装 MindSpore NLP 0.5.1（示例：从 PyPI 安装）
# !pip install mindnlp==0.5.1 -i https://pypi.tuna.tsinghua.edu.cn/simple

# 若需要安装 HuggingFace datasets 和 evaluate：
# !pip install datasets evaluate tqdm -i https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
import os
import numpy as np
from tqdm import tqdm

import mindspore as ms
from mindspore import context

from mindnlp.dataset import load_dataset
from mindnlp.transformers import AutoTokenizer, AutoModelForCausalLM

print("MindSpore version:", ms.__version__)

import mindnlp
print("MindNLP version:", mindnlp.__version__)

#### **设置 MindSpore 上下文**

Ascend（Atlas）设备，默认使用PYNATIVE_MODE模式

In [ ]:
device_target = os.getenv("DEVICE_TARGET", "Ascend")  # 如需在CPU上运行，将"Ascend"改为"CPU"
print("Using device:", device_target)

context.set_context(
    device_target=device_target
)


## 数据加载与预处理

我们使用 HuggingFace 上的 **wikitext-2-raw-v1** 数据集作为语言模型训练语料。
该数据集包含维基百科条目文本，是语言模型常用的开源基准数据集之一。

MindSpore NLP 提供了 `load_dataset` 接口，可直接从 HuggingFace Datasets 仓库拉取数据。

#### **加载 Wikitext-2-raw-v1 数据集**

In [ ]:
# 分别加载 train / validation / test 三个划分
# 这里指定子集名称 'wikitext-2-raw-v1'
wiki_ds_dict = load_dataset(
    "wikitext",
    name="wikitext-2-raw-v1",
    split=["train", "validation", "test"]
)

train_raw = wiki_ds_dict["train"]
valid_raw = wiki_ds_dict["validation"]
test_raw  = wiki_ds_dict["test"]

print("Train size:", len(train_raw))
print("Valid size:", len(valid_raw))
print("Test size:", len(test_raw))

# 查看一个样本
print("\nExample sample from train:")
print(train_raw[10])

## 模型构建

#### **加载预训练模型与分词器（DistilGPT-2）**

In [ ]:
model_name = "distilgpt2"  # 也可使用 "gpt2" 等其他 Causal LM 模型

# 若使用国内镜像，可根据平台设置环境变量 HF_ENDPOINT 或使用 mirror/modelscope 参数
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Tokenizer vocab size:", len(tokenizer))
print("Model loaded:", type(model))

# GPT-2 家族默认没有 pad_token，这里将 pad_token 设置为 eos_token，方便批量 padding
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 调整模型的词表大小以适配分词器（例如添加了 pad_token 的情况）
model.resize_token_embeddings(len(tokenizer))

#### **文本预处理与语言建模数据构造**

对于自回归语言模型（Causal LM），训练数据通常形如：

- 输入：`input_ids = [w_1, w_2, ..., w_{n-1}]`
- 标签：`labels = [w_2, w_3, ..., w_n]`

在大多数 GPT 类实现中，可以**直接令 `labels` 与 `input_ids` 相同**，
模型内部在计算 loss 时会自动进行「右移一位」的处理，并忽略 padding 位置的标签（常用 `-100`）。

本实验中，我们做如下简化处理：

1. 对每条文本单独进行 token 化与截断（不做跨样本拼接）；
2. 令 `labels = input_ids.copy()`，pad 时对 `labels` 使用填充值 `-100`，以避免影响 loss；
3. 使用 MindSpore 数据集的 `padded_batch` 完成动态 padding。

In [ ]:
import mindspore.dataset as ds
from mindspore.dataset import transforms

max_seq_len = 128   # 可根据显存调整
train_batch_size = 64
eval_batch_size = 64

def _to_py_str(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="ignore")
    if isinstance(x, (np.bytes_,)):
        return x.decode("utf-8", errors="ignore")
    if isinstance(x, (np.str_,)):
        return str(x)
    if isinstance(x, np.ndarray):
        if x.ndim == 0:
            return _to_py_str(x.item())
        return " ".join(_to_py_str(t) for t in x.tolist())
    try:
        if isinstance(x, ms.Tensor):
            return _to_py_str(x.asnumpy())
    except Exception:
        pass
    return str(x)

def process_lm_dataset(dataset,
                       tokenizer,
                       max_seq_len=128,
                       batch_size=64,
                       shuffle=False,
                       take_len=None):
    # GPT-2 系确保有 pad_token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # 打乱 / 截取
    if shuffle:
        dataset = dataset.shuffle(buffer_size=batch_size * 100)
    if take_len:
        dataset = dataset.take(take_len)

    # 文本标准化
    dataset = dataset.map(
        operations=[_to_py_str],
        input_columns="text",
        output_columns=["text"],
        num_parallel_workers=4,
    )

    # 分词 + labels（保证至少 1 个 token）
    def tokenize_and_create_labels(text):
        text = _to_py_str(text)
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_len,
            add_special_tokens=True,
        )
        ids = tokenized["input_ids"]
        if len(ids) == 0:
            ids = [tokenizer.eos_token_id]
        input_ids = np.array(ids, dtype=np.int32)
        labels = input_ids.copy()
        return input_ids, labels

    dataset = dataset.map(
        operations=[tokenize_and_create_labels],
        input_columns="text",
        output_columns=["input_ids", "labels"],
        num_parallel_workers=4,
    )

    # 显式类型
    type_cast_op = transforms.TypeCast(ms.int32)
    dataset = dataset.map(operations=type_cast_op, input_columns="input_ids", num_parallel_workers=4)
    dataset = dataset.map(operations=type_cast_op, input_columns="labels",    num_parallel_workers=4)

    # 只保留数值列
    dataset = dataset.project(["input_ids", "labels"])

    # 定长 padded_batch，避免 None 形状带来的歧义/兼容性问题
    dataset = dataset.padded_batch(
        batch_size=batch_size,
        pad_info={
            "input_ids": ([max_seq_len], tokenizer.pad_token_id),
            "labels":    ([max_seq_len], -100),
        },
        drop_remainder=False,  # 需要的话可以改成 True
    )
    return dataset

In [ ]:
train_take_len = 2000
valid_take_len = 512
test_take_len  = 512

train_dataset = process_lm_dataset(
    train_raw,
    tokenizer,
    max_seq_len=max_seq_len,
    batch_size=train_batch_size,
    shuffle=True,
    take_len=train_take_len
)

valid_dataset = process_lm_dataset(
    valid_raw,
    tokenizer,
    max_seq_len=max_seq_len,
    batch_size=eval_batch_size,
    shuffle=False,
    take_len=valid_take_len
)

test_dataset = process_lm_dataset(
    test_raw,
    tokenizer,
    max_seq_len=max_seq_len,
    batch_size=eval_batch_size,
    shuffle=False,
    take_len=test_take_len
)

print("Train dataset size (batches):", train_dataset.get_dataset_size())
print("Valid dataset size (batches):", valid_dataset.get_dataset_size())
print("Test dataset size (batches):",  test_dataset.get_dataset_size())

for batch in train_dataset.create_dict_iterator():
    print("input_ids shape:", batch["input_ids"].shape)
    print("labels shape:", batch["labels"].shape)
    break

### 模型训练

为了简化训练流程并利用 Hugging Face 风格的 API，我们采用 `mindnlp.transformers.Trainer` 进行训练。针对 MindSpore 后端的特性，我们对 Trainer 进行了如下关键适配：

1. **数据接口适配 (`MSMapDataset`)**
   - 将 MindSpore 原生的流式 Dataset（Iterable）封装为支持下标访问的 Map-style 数据集。
   - 将 Batch 数据预先缓存为 NumPy 格式，以便 Trainer 内部的 DataLoader 能正确索引和分发。
2. **自定义 Trainer (`NoJitTrainer`)**
   - 继承自 `mindnlp` 的 `Trainer`，主要为了解决动态图模式下的梯度计算问题。
   - **重写 `training_step`**：移除默认的 JIT 编译（静态图加速），采用显式的 `.backward()` 反向传播，避免 `value_and_grad` 在复杂控制流下的潜在兼容性报错。
   - **重写 `compute_loss`**：增加了对 `loss_type="ForCausalLMLoss"` 的强制检查，并提供手动计算 CrossEntropy 的兜底逻辑，确保在模型输出不含 loss 字段时也能正常训练。
3. **数据整理 (`passthrough_collator`)**
   - 实现了一个直通式 Collator，负责将数据转换为 `mindtorch` Tensor（或 `int64` 类型的 Numpy 数组），不指定具体 Device，交由 Trainer 自动管理设备放置。
4. **训练配置**
   - 使用 `TrainingArguments` 管理超参，配置 `adamw_torch` 优化器，设置 `learning_rate=5e-5`，并开启评估模式 (`do_eval=True`)。


In [ ]:
# ===== 使用 MindSpore NLP Trainer 训练 =====
import math
import numpy as np
import mindspore as ms
from mindspore import context
from mindnlp.transformers import TrainingArguments, Trainer as _BaseTrainer
from transformers.trainer_callback import TrainerCallback

# 将 MindSpore Dataset（已 padded_batch 的“批”）封成可下标 map-style
class MSMapDataset:
    """把 MindSpore Dataset 的每个 batch 缓存为 numpy，供 HF Trainer 索引。"""
    def __init__(self, ms_dataset):
        self.cache = []
        for b in ms_dataset.create_dict_iterator():
            def to_np(x):
                return x.asnumpy() if hasattr(x, "asnumpy") else np.asarray(x)
            self.cache.append({
                "input_ids": to_np(b["input_ids"]),
                "labels":    to_np(b["labels"]),
            })
    def __len__(self):
        return len(self.cache)
    def __getitem__(self, idx):
        return self.cache[idx]

train_map = MSMapDataset(train_dataset)
valid_map = MSMapDataset(valid_dataset)

# 直通式 collator：不做二次 padding/batch，不指定 device，直接产 int64
def passthrough_collator(features):
    feat = features[0] if isinstance(features, list) and len(features) == 1 else features
    arr_ids, arr_lbl = feat["input_ids"], feat["labels"]

    # 优先用 mindtorch Tensor；不指定 device，避免 'Ascend' 相关报错
    try:
        import mindtorch as mt
        return {
            "input_ids": mt.tensor(arr_ids, dtype=mt.int64),
            "labels":    mt.tensor(arr_lbl, dtype=mt.int64),
        }
    except Exception:
        # 兜底：numpy（Trainer 也能吃）
        return {
            "input_ids": np.asarray(arr_ids, dtype=np.int64),
            "labels":    np.asarray(arr_lbl, dtype=np.int64),
        }

# 自定义 Trainer：手写 backward，避免 value_and_grad / JIT
class NoJitTrainer(_BaseTrainer):
    def __init__(self, *args, loss_type: str = "ForCausalLMLoss", **kwargs):
        # 收下想用的 loss_type
        self._force_loss_type = loss_type

        # 在父类 __init__ 之前尽早写入 config.loss_type，避免早期 warning
        model = kwargs.get("model", args[0] if len(args) > 0 else None)
        if model is not None and hasattr(model, "config"):
            try:
                if getattr(model.config, "loss_type", None) != self._force_loss_type:
                    setattr(model.config, "loss_type", self._force_loss_type)
            except Exception:
                pass

        super().__init__(*args, **kwargs)

    def _ensure_loss_type(self, model):
        cfg = getattr(model, "config", None)
        if cfg is not None and self._force_loss_type:
            try:
                if getattr(cfg, "loss_type", None) != self._force_loss_type:
                    setattr(cfg, "loss_type", self._force_loss_type)
            except Exception:
                pass

    # 补上 num_items_in_batch 和 **kwargs 以兼容 Trainer 调用
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        self._ensure_loss_type(model)

        outputs = model(**inputs)
        loss = getattr(outputs, "loss", None)

        if loss is None:
            # 兜底：手动 CE（shift + ignore_index=-100）
            import mindtorch as mt
            import mindtorch.nn.functional as F
            logits = outputs.logits                  # [B, L, V]
            labels = inputs["labels"]                # [B, L]
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1),
                ignore_index=-100
            )

        return (loss, outputs) if return_outputs else loss

    # ⚠️ 同样把签名补齐，Trainer 有时也会传这个参数
    def training_step(self, model, inputs, num_items_in_batch=None, **kwargs):
        model.train()
        loss = self.compute_loss(model, inputs, num_items_in_batch=num_items_in_batch)

        if self.args.gradient_accumulation_steps > 1:
            loss = loss / self.args.gradient_accumulation_steps

        loss.backward()
        return loss.detach()


# 配置 TrainingArguments
num_epochs = 3  # 演示可 1~3
training_args = TrainingArguments(
    output_dir="./outputs/gpt2_wikitext2_ms",
    num_train_epochs=num_epochs,
    learning_rate=5e-5,
    optim="adamw_torch",          # 使用合法枚举，避免 'adamw' 报错
    logging_steps=max(1, len(train_map)//20),   # 每个 epoch 约打印 20 次 step 级 loss

    # 上游已 padded_batch，这里每步就取“一个批”
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    dataloader_num_workers=0,
    remove_unused_columns=False,
    do_eval=True,                 # 打开评估（我们用回调强制每个 epoch 评一次）
)

trainer = NoJitTrainer(
    model=model,
    args=training_args,
    train_dataset=train_map,
    eval_dataset=valid_map,
    data_collator=passthrough_collator,
    loss_type="ForCausalLMLoss",
)

#### **开始训练**

为节省时间，我们这里演示性地训练 **3～5 个 epoch**。实际任务中可按需加大训练轮数。

训练过程中会打印：

- 每个 epoch 的平均训练 loss；
- 每个 epoch 结束后的验证集平均 loss。

In [ ]:
# ===== 启动训练：打印每个 epoch 的平均 Train/Valid Loss =====

class EpochAvgAndEval(TrainerCallback):
    """收集 step 级 loss；在每个 epoch 末强制 evaluate，并打印平均训练 loss 与验证 loss。"""
    def __init__(self):
        self._loss_buf = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            try:
                self._loss_buf.append(float(logs["loss"]))
            except Exception:
                pass

    def on_epoch_end(self, args, state, control, **kwargs):
        ep = int(state.epoch) if state.epoch is not None else -1
        if self._loss_buf:
            avg_train = sum(self._loss_buf) / len(self._loss_buf)
            print(f"Train loss (epoch {ep}): {avg_train:.4f}")
        else:
            print(f"Train loss (epoch {ep}): N/A")
        self._loss_buf.clear()

        # 强制本 epoch 末评估（即使没有 evaluation_strategy）
        control.should_evaluate = True
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        ep = int(state.epoch) if state.epoch is not None else -1
        if metrics and metrics.get("eval_loss") is not None:
            print(f"Valid  loss (epoch {ep}): {metrics['eval_loss']:.4f}")

# 挂回调
trainer.add_callback(EpochAvgAndEval())

# （可选）也可以在开始前先做一次 quick sanity eval
# _ = trainer.evaluate()

# 开始训练（每个 epoch 末都会打印 Train/Valid loss）
train_output = trainer.train()

# 训练结束后再评一次并打印 PPL
final_metrics = trainer.evaluate()
if final_metrics.get("eval_loss") is not None:
    ppl = math.exp(final_metrics["eval_loss"])
    print(f"Final Eval loss: {final_metrics['eval_loss']:.4f} | PPL: {ppl:.2f}")
else:
    print("Final eval_loss 不存在，跳过 PPL 计算。")

## 模型推理

训练完成后，我们使用 `model.generate` 进行文本自动续写。
整体流程：

1. 准备一个中文或英文的起始提示（prompt）；
2. 使用分词器编码为 `input_ids`；
3. 调用 `model.generate` 生成若干新 token；
4. 用分词器解码为可读文本。

In [ ]:
def generate_text(model,
                  tokenizer,
                  prompt,
                  max_new_tokens=50,
                  do_sample=True,
                  top_p=0.9,
                  temperature=1.0):
    import numpy as np
    import mindtorch as mt

    # 以模型参数的 device 为准，避免 device 不一致
    try:
        model_device = next(model.parameters()).device
    except Exception:
        model_device = mt.device("cpu")  # 极端兜底

    # eval 模式
    model.set_train(False)  # 或 model.eval()

    # pad_token 兜底（GPT-2 常见）
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    # 编码 -> 放到“模型的 device”
    enc = tokenizer(prompt, add_special_tokens=True)
    input_ids = mt.tensor(np.array([enc["input_ids"]]), device=model_device).long()

    # 生成
    with mt.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            top_p=top_p,
            temperature=temperature,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # 解码
    generated_ids = outputs[0].tolist()
    return tokenizer.decode(generated_ids, skip_special_tokens=True)




## 试一试生成

prompt = "Deep learning has changed natural language processing because"
print("Prompt:\n", prompt)

generated = generate_text(
    model, tokenizer, prompt,
    max_new_tokens=60,
    do_sample=True,
    top_p=0.95,
    temperature=0.8
)
print("\nGenerated text:\n", generated)

## 模型保存与加载（可选）

为了在后续 Notebook 或部署场景中复用微调结果，可以将模型权重与分词器信息保存到本地目录，
之后通过 `from_pretrained` 的方式重新加载。

In [ ]:
import os

save_dir = "./distilgpt2-ms-finetuned-wikitext2"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Model & tokenizer saved to:", save_dir)

## 加载（验证）

from mindnlp.transformers import AutoTokenizer, AutoModelForCausalLM

loaded_tokenizer = AutoTokenizer.from_pretrained(save_dir)
loaded_model = AutoModelForCausalLM.from_pretrained(save_dir)

test_prompt = "Language models are"
generated_loaded = generate_text(loaded_model, loaded_tokenizer, test_prompt)
print("\n[Loaded model generation]\n", generated_loaded)